In [ ]:
# Cell 1: Setup

import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor

from bufferiq.ml.optimization.optuna_optimizer import OptunaOptimizer
from bufferiq.ml.optimization.optuna_samplers import SamplerRegistry
from bufferiq.ml.optimization.optuna_pruners import PrunerRegistry
from bufferiq.ml.optimization.multi_objective import MultiObjectiveOptimizer
from bufferiq.ml.optimization.param_importance import HyperparameterImportanceAnalyzer
from bufferiq.ml.optimization.study_manager import OptunaStudyManager
from bufferiq.ml.optimization.advanced_visualizer import AdvancedOptimizationVisualizer

In [ ]:
# Cell 2: Load Data

df = pd.read_csv('../outputs/features/train_features.csv')

X = df.drop(columns=['engagement_score']).values
y = df['engagement_score'].values

print(f"Data shape: X={X.shape}, y={y.shape}")

In [ ]:
# Cell 3: Basic Optuna Optimization

search_space = {
    'learning_rate': {'type': 'float', 'low': 0.01, 'high': 0.3, 'log': True},
    'max_depth': {'type': 'int', 'low': 3, 'high': 10},
    'n_estimators': {'type': 'int', 'low': 100, 'high': 500, 'step': 50},
    'subsample': {'type': 'float', 'low': 0.6, 'high': 1.0},
    'colsample_bytree': {'type': 'float', 'low': 0.6, 'high': 1.0},
}

model = XGBRegressor(random_state=42)

sampler = SamplerRegistry.get_sampler('tpe', seed=42)
pruner = PrunerRegistry.get_pruner('median')

optimizer = OptunaOptimizer(
    model=model,
    search_space=search_space,
    n_trials=50,
    sampler=sampler,
    pruner=pruner,
    study_name='notebook_demo',
    storage='sqlite:///../outputs/notebook_study.db'
)

results = optimizer.search(X, y)

print(f"\nBest R²: {results['best_score']:.4f}")
print(f"Best params: {results['best_params']}")
print(f"Trials: {results['n_trials']} (pruned: {results['n_pruned']})")

In [ ]:
# Cell 4: Compare Pruning Strategies

import time

pruner_results = {}

for pruner_name in ['nop', 'median', 'hyperband']:
    print(f"\nTesting {pruner_name} pruner...")
    
    pruner = PrunerRegistry.get_pruner(pruner_name)
    
    optimizer = OptunaOptimizer(
        model=model,
        search_space=search_space,
        n_trials=30,
        sampler=sampler,
        pruner=pruner,
        study_name=f'pruner_{pruner_name}',
        storage=f'sqlite:///../outputs/pruner_{pruner_name}.db'
    )
    
    start_time = time.time()
    results = optimizer.search(X, y)
    duration = time.time() - start_time
    
    pruner_results[pruner_name] = {
        'best_score': results['best_score'],
        'n_pruned': results['n_pruned'],
        'duration': duration
    }

comparison_df = pd.DataFrame(pruner_results).T
print("\nPruner Comparison:")
print(comparison_df)

In [ ]:
# Cell 5: Multi-Objective Optimization

multi_opt = MultiObjectiveOptimizer(
    model=model,
    search_space=search_space,
    metrics=['r2', 'training_time', 'model_size'],
    directions=['maximize', 'minimize', 'minimize'],
    n_trials=50,
    cv=5
)

multi_results = multi_opt.search(X, y)

print(f"\nFound {multi_results['n_pareto_solutions']} Pareto solutions")

multi_opt.visualize_pareto_front(
    Path('../outputs/notebook_pareto.html')
)

print("Pareto front saved to ../outputs/notebook_pareto.html")

In [ ]:
# Cell 6: Hyperparameter Importance Analysis

analyzer = HyperparameterImportanceAnalyzer(results['study'])
importance = analyzer.calculate_importance()

analyzer.visualize_importance(
    importance,
    Path('../outputs/notebook_importance.png')
)

print("\nHyperparameter Importance:")
for rank, (param, score) in enumerate(
    sorted(importance.items(), key=lambda x: x[1], reverse=True), 1
):
    print(f"  {rank}. {param}: {score:.4f}")

In [ ]:
# Cell 7: Study Management

manager = OptunaStudyManager('sqlite:///../outputs/notebook_study.db')

studies = manager.list_studies()
print(f"\nFound {len(studies)} studies:")

for study_name in studies:
    summary = manager.get_study_summary(study_name)
    print(f"  - {study_name}: {summary['n_trials']} trials, best={summary['best_value']:.4f}")

if studies:
    manager.export_study(
        studies[0],
        Path('../outputs/notebook_study_export.json')
    )
    print("\nExported study to ../outputs/notebook_study_export.json")

In [ ]:
# Cell 8: Advanced Visualizations

visualizer = AdvancedOptimizationVisualizer(results['study'])

output_dir = Path('../outputs/notebook_visualizations')
visualizer.create_all_visualizations(output_dir)

print(f"\nAll visualizations saved to {output_dir}")
print("Available plots:")

for plot_file in output_dir.glob('*.html'):
    print(f"  - {plot_file.name}")

In [ ]:
# Cell 9: Resume Interrupted Study

study = manager.load_study('notebook_demo')
print(f"Loaded study with {len(study.trials)} existing trials")

# Uncomment to continue optimization
# study.optimize(optimizer._objective, n_trials=20)

In [ ]:
# Cell 10: Summary

print("""
Key Takeaways:

1. Optuna provides efficient hyperparameter optimization
2. Pruning significantly reduces computation time
3. Multi-objective optimization finds trade-offs
4. Learning rate and max_depth are most impactful

Next Steps:
- Increase trials (200+)
- Try different samplers (CMA-ES)
- Validate on holdout set
- Deploy best model
""")